# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

Ce notebook permet d'exécuter l'entraînement distribué par Deep Reinforcement Learning (PPO & Self-Play) directement depuis **VS Code** (via l'extension Google Colab ou votre kernel local) ou sur **Google Colab web**.

## 1. Détection de l'Environnement et Installation des Dépendances

In [ ]:
import os, sys

# 1. Récupération des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Récupération des dernières mises à jour du repo...')
    !git pull origin main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et mise à jour...')
    %cd agario
    !git pull origin main
else:
    print('🌐 Environnement distant Colab détecté. Clonage du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip install -q -r requirements.txt tensorboard

# 3. Vérification GPU CUDA
import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU actif : {torch.cuda.get_device_name(0)}')
else:
    print('Exécution sur CPU.')


## 2. Validation des Tests Unitaires (20 Tests)

In [10]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/agario
configfile: pytest.ini
testpaths: tests
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 2 items / 4 errors                                                   

==================================== ERRORS ====================================
________________ ERROR collecting tests/test_engine_physics.py _________________
ImportError while importing test module '/content/agario/tests/test_engine_physics.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_engine_physics.py:7: in <module>
    from src.env.agar_engine import AgarEngine


## 3. Monitoring TensorBoard

In [11]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

## 4. Démarrer l'Entraînement PPO & Self-Play

In [12]:
# Lance 16 environnements en parallèle avec 10 bots par arène et mise à jour du pool d'adversaires
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 1000000 \
    --batch-size 128 \
    --n-steps 2048 \
    --pool-interval 200000 \
    --device auto

2026-09-23 21:56:00.818579: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Traceback (most recent call last):
  File "/content/agario/src/training/train_colab.py", line 26, in <module>
    from src.env.gym_wrapper import AgarEnv
ModuleNotFoundError: No module named 'src.env'


## 5. Exporter la Politique vers ONNX (< 0.02 ms de latence CPU)

In [13]:
!python src/inference/export_onnx.py \
    --model checkpoints/ppo/ppo_final.zip \
    --output models/model.onnx

2026-09-23 21:56:09.116232: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Error: Model file 'checkpoints/ppo/ppo_final.zip' not found.


## 6. Téléchargement du Modèle (si exécuté sur VM Colab distante)

In [14]:
try:
    from google.colab import files
    files.download('models/model.onnx')
    files.download('checkpoints/ppo/ppo_final.zip')
    print("Téléchargement Colab initié.")
except ImportError:
    print("Fichiers sauvegardés localement dans models/ et checkpoints/.")

FileNotFoundError: Cannot find file: models/model.onnx